In [1]:
# Run this in a new Colab cell
!pip install transformers datasets evaluate torch torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00


In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoImageProcessor, ResNetForImageClassification, TrainingArguments, Trainer
import evaluate
import numpy as np
from PIL import Image

# --- 1. LOAD THE DATA ---
# Hugging Face's "imagefolder" automatically labels images based on their sub-folder names!
# Ensure your folders are: image_data/disaster/ and image_data/safe/
dataset = load_dataset("imagefolder", data_dir="image_data")

# Split into 80% training and 20% testing
dataset = dataset["train"].train_test_split(test_size=0.2)

# --- 2. THE IMAGE PROCESSOR (Image -> Numbers) ---
model_name = "microsoft/resnet-50"
processor = AutoImageProcessor.from_pretrained(model_name)

def process_images(batch):
    # 1. Convert to RGB (in case some downloaded images are grayscale or PNGs with transparency)
    images = [img.convert("RGB") for img in batch["image"]]
    # 2. The processor resizes them to 224x224 and normalizes the pixel colors
    inputs = processor(images, return_tensors="pt")
    # 3. Attach the labels
    inputs["labels"] = batch["label"]
    return inputs

# .with_transform applies the processing on-the-fly so we don't crash Colab's RAM
processed_dataset = dataset.with_transform(process_images)

# --- 3. THE MODEL ---
# Define our labels so the model outputs human-readable text later
id2label = {0: "safe", 1: "disaster"}
label2id = {"safe": 0, "disaster": 1}

# Load ResNet-50. 
# ignore_mismatched_sizes=True is REQUIRED because we are replacing ResNet's 
# original 1,000-category brain with our custom 2-category brain!
model = ResNetForImageClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True 
)

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return metric.compute(predictions=predictions, references=eval_pred.label_ids)

# --- 4. TRAINING SETUP ---
# A "Collator" is a mini-function that stacks our individual 224x224 images into batches
def collate_fn(examples):
    pixel_values = torch.stack([example["pixel_values"] for example in examples])
    labels = torch.tensor([example["labels"] for example in examples])
    return {"pixel_values": pixel_values, "labels": labels}

training_args = TrainingArguments(
    output_dir="./resnet_results",
    eval_strategy="epoch",
    learning_rate=5e-5,          
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    remove_unused_columns=False, # <-- CRITICAL FOR IMAGES! Prevents HF from deleting image data
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=processed_dataset["train"],
    eval_dataset=processed_dataset["test"],
    compute_metrics=compute_metrics,
    data_collator=collate_fn,
)

# --- 5. TRAIN IT! ---
print("Starting ResNet training...")
trainer.train()

# --- 6. TEST IT ON A NEW IMAGE ---
def predict_image(image_path):
    # Open the image using the PIL library
    image = Image.open(image_path).convert("RGB")
    
    # Process it exactly like we did during training
    inputs = processor(image, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model(**inputs)
        
    prediction = torch.argmax(outputs.logits, dim=-1).item()
    confidence = torch.softmax(outputs.logits, dim=-1).max().item()
    
    label = model.config.id2label[prediction]
    print(f"Prediction for {image_path}: {label} (Confidence: {confidence*100:.1f}%)")

# Upload a random test image to Colab and try it!
# predict_image("my_test_fire_photo.jpg")